# Artifact parser test

Interactive checks for `shared.artifacts.ArtifactHandler`.

Parse returns **addressable blocks**: agents cite `block_id`; a viewer can open `page` and highlight `bbox`.

PDF pages are classified then reconstructed. Same JSON contract; `extra.reconstructor` is debug-only.

| `pages[].kind` | reconstructor |
|---|---|
| `digital`, `mixed` | `letter` (column-aware) |
| `dashboard` | `dashboard` (KPI tiles) |
| `table` | `grid` |
| `design` | `cover` |
| `chart` | `chart` |
| `scanned` | `scanned` |

Supported files: `.pdf`, `.pptx`, `.docx`, `.xlsx` / `.xlsm`.

1. Set `FILE_PATH` in the config cell.
2. Run **classify**, then **parse**.
3. Inspect `to_dict()` plus page `kind`, per-page block counts, and `extra.reconstructor`.

In [ ]:
from __future__ import annotations

import json
import os
import re
import sys
from collections import Counter, defaultdict
from pathlib import Path

from IPython.display import Markdown, display

cwd = Path.cwd().resolve()
repo_root = next(
    (p for p in [cwd, *cwd.parents] if (p / "shared" / "artifacts").is_dir()),
    cwd,
)
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from shared.artifacts import ArtifactHandler, ParseOptions, citation_from_block
from shared.artifacts.exceptions import ArtifactError

print(f"repo_root = {repo_root}")

## Config

Point `FILE_PATH` at a local pdf / pptx / docx / xlsx. Leave `ARTIFACT_TYPE` as `None` to classify first.

PDF image/chart blocks are emitted structurally (not gated on `include_images`). Set `use_ocr=True` only for scanned pages.

For a dashboard/grid PDF (Graydon-style), page 1 should be `dashboard` and KPI labels should share a `text` block with their values. Letter PDFs (BDO) stay `digital` / `mixed`.

In [ ]:
FIXTURES = Path(os.environ.get("ARTIFACT_FIXTURES_DIR", r"D:\Work\Etex\Simple example"))
FILE_PATH = FIXTURES / "KPMG_ADVISORY_SRL_BE-X-439819279_2026-03-18_14-39-57.pdf"
ARTIFACT_TYPE = None  # or "pdf" | "ppt" | "excel" | "word"
PARSE_OPTIONS = ParseOptions(
    password=None,
    include_tables=True,
    include_images=False,
    include_hidden_sheets=False,
    max_pages=None,
    use_ocr=False,
)

SAVE_JSON = repo_root / "test_files" / "parser_test" / "sample_output.json"
SAVE_MD = repo_root / "test_files" / "parser_test" / f"{FILE_PATH.stem}.md"

path = FILE_PATH.expanduser().resolve()
print(f"exists={path.is_file()}  path={path}")
print(f"save json={SAVE_JSON}")
print(f"save md={SAVE_MD}")

## Classify

In [ ]:
handler = ArtifactHandler()
kind = handler.classify(path)
print(json.dumps({"path": str(path), "action": "classify", "artifact_type": kind.value}, indent=2))

## Parse

`document.to_dict()` is the JSON contract: `artifact_id`, `coord_system`, `pages`, `outline`, `blocks`, `warnings`, `markdown`. No `action` / `plain_text` / `MD_text` keys.

PDF `pages[]` include `kind`, `rotation`, and optional `mediabox` when it differs from crop. Bboxes stay crop-relative (`pdf_points_top_left`).

In [ ]:
try:
    document = handler.parse(path, ARTIFACT_TYPE, options=PARSE_OPTIONS)
except ArtifactError as exc:
    raise SystemExit(f"parse failed: {exc}") from exc

payload = document.to_dict()
print(
    f"type={document.artifact_type.value}  artifact_id={document.artifact_id}  "
    f"coord={document.coord_system}  pages={len(document.pages)}  "
    f"blocks={len(document.blocks)}  outline={len(document.outline)}"
)
print("warnings:", document.warnings or "none")
print("to_dict keys:", list(payload))
print("page kinds:", dict(Counter(page.kind for page in document.pages)))
mediabox_pages = [page.page for page in document.pages if page.mediabox]
print("pages with mediabox:", mediabox_pages or "none (crop == media)")
print(json.dumps(document.metadata.to_dict(), indent=2, default=str))

## Block summary

`id` is a debug sequence. Agents cite **`block_id`**. `heading_path` is the section stack at that block.

`extra.reconstructor` and `extra.page_kind` are debug fields on each block.

In [ ]:
print("types:", dict(Counter(block.type.value for block in document.blocks)))
print(
    "reconstructors:",
    dict(Counter(block.extra.get("reconstructor", "?") for block in document.blocks)),
)
print("unique block_id:", len({block.block_id for block in document.blocks}) == len(document.blocks))
print()
for block in document.blocks[:20]:
    loc = block.location.to_dict()
    where = {k: loc[k] for k in ("page", "slide", "sheet", "bbox") if k in loc}
    preview = block.text.replace("\n", " ")[:90]
    heading = " > ".join(block.heading_path[-2:]) if block.heading_path else ""
    rec = block.extra.get("reconstructor", "")
    print(
        f"{block.id:12} {block.block_id:16} {block.type.value:8} {rec:10} {where}  {heading}  {preview}"
    )

## Page dispatch

Classifier `kind` vs reconstructor vs block count. Dashboard/grid pages should not explode into hundreds of chips. Cover pages must not be empty.

In [ ]:
by_page: dict[int | None, list] = defaultdict(list)
for block in document.blocks:
    by_page[block.location.page].append(block)

print(f"{'page':>4}  {'kind':10}  {'blocks':>6}  types  reconstructors")
for page in document.pages:
    blocks = by_page.get(page.page, [])
    types = dict(Counter(block.type.value for block in blocks))
    recs = dict(Counter(block.extra.get("reconstructor", "?") for block in blocks))
    print(f"{page.page:4}  {page.kind:10}  {len(blocks):6}  {types}  {recs}")

## Quality checks

Chrome (`Printed On`, `Page N of N`, `ETEX GROUP I`) must not land in `block.text`. Dates / postcodes / `ESG` / numeric ranges should not be headings. KPI tiles (if present) stay one `text` block.

In [ ]:
chrome_hits = [
    block.text.strip()[:80]
    for block in document.blocks
    if re.search(
        r"(?i)(printed on|^page\s+\d+\s+of\s+\d+$|^etex group i)",
        block.text.strip(),
    )
]
print("chrome in body:", chrome_hits[:8] or "none")

headings = [block.text.split("\n", 1)[0][:70] for block in document.blocks if block.type.value == "heading"]
print(f"headings ({len(headings)}):")
for title in headings[:15]:
    print(f"  - {title}")

credit = next((block for block in document.blocks if "Credit Limit" in block.text), None)
if credit is None:
    print("credit tile: (not in this file)")
else:
    compact = credit.text.replace(" ", "").replace(",", "").replace(".", "")
    print(
        f"credit tile: type={credit.type.value} reconstructor={credit.extra.get('reconstructor')} "
        f"has_amount={'1750000' in compact} text={credit.text[:160]!r}"
    )

woven = [
    block.text[:100]
    for block in document.blocks
    if "+32" in block.text and "Registration" in block.text
]
print("column weave (phone + Registration Number):", woven[:3] or "none")

## Citation helper

Not parser output. `citation_from_block(...)` is what an agent can return so a viewer can open the box.

Prefers a BDO-style fee table (`35.251`), else the first table, else the Credit Limit tile, else the first block.

In [ ]:
fee = next(
    (
        block
        for block in document.blocks
        if block.type.value == "table" and "35.251" in (block.text or "")
    ),
    None,
)
target = fee or next((block for block in document.tables()), None)
if target is None:
    target = next((block for block in document.blocks if "Credit Limit" in block.text), None)
if target is None:
    target = document.blocks[0]

citation = citation_from_block(target, document.artifact_id, document.coord_system)
print("cited", target.id, target.type.value, "reconstructor=", target.extra.get("reconstructor"))
print(json.dumps(citation, indent=2, ensure_ascii=False))
print("heading_path:", target.heading_path)

## Save Markdown

Writes `document.md_text()` to a `.md` file next to this notebook. Location is an HTML comment only: `<!-- loc page=N block=b_xxx type=table -->`.

In [ ]:
markdown = document.md_text()
if markdown and not markdown.endswith("\n"):
    markdown += "\n"
SAVE_MD.parent.mkdir(parents=True, exist_ok=True)
SAVE_MD.write_text(markdown, encoding="utf-8")
print(f"wrote {SAVE_MD}  ({SAVE_MD.stat().st_size:,} bytes)")
display(Markdown(markdown[:4000] + ("\n\n…" if len(markdown) > 4000 else "")))

## Loc comments

Same string written to the `.md` file / `document.to_dict()["markdown"]`. Filename belongs in YAML only, not in body text.

In [ ]:
locs = [line for line in markdown.splitlines() if line.startswith("<!-- loc ")]
print(f"loc comments: {len(locs)}")
print("\n".join(locs[:12]))
print()
body = markdown.split("---", 2)[-1]
print("filename in body:", document.metadata.filename in body)

## Save JSON

Writes `document.to_dict()` (the parse contract). Same payload as `test.py … parse`.

In [ ]:
SAVE_JSON.parent.mkdir(parents=True, exist_ok=True)
SAVE_JSON.write_text(
    json.dumps(payload, indent=2, ensure_ascii=False, default=str),
    encoding="utf-8",
)
print(f"wrote {SAVE_JSON}  ({SAVE_JSON.stat().st_size:,} bytes)")